In [1]:
#organize the isd data by date-time
#get the numeric data from json columns

import pandas as pd
prop_census_isd = pd.read_csv("properties_census_isd.csv")
#print all values in column that indicates census tract
print("Columns:")
print(prop_census_isd.columns.tolist())
tract_list = prop_census_isd['tract'].tolist()


Columns:
['rs_prop_sam_id', 'rs_prop_prim_sam_id', 'rs_prop_full_address', 'rs_prop_street_number', 'rs_prop_full_street_name', 'rs_prop_unit_number', 'rs_prop_zip_code', 'rs_prop_mailing_neighborhood', 'rs_prop_long', 'rs_prop_lat', 'rs_prop_pid', 'rs_prop_lu', 'rs_prop_owner', 'rs_prop_yr_built', 'rs_prop_yr_remod', 'rs_prop_living_area', 'rs_prop_bdrms', 'rs_prop_full_bth', 'rs_prop_half_bth', 'json_isd_housing_5year', 'json_bldg_isd_housing_5year', 'json_isd_building_5year', 'json_bldg_isd_building_5year', 'json_isd_code_5year', 'json_bldg_isd_code_5year', 'json_mhl_civic_5year', 'json_bldg_mhl_civic_5year', 'json_mhl_housing_5year', 'json_bldg_mhl_housing_5year', 'json_mhl_sanitation_5year', 'json_bldg_mhl_sanitation_5year', 'tract', 'Housing_Vehicles_share_no_vehicles', 'Housing_LivingArrangements_household_pop', 'Housing_LivingArrangements_group_quarters', 'Housing_LivingArrangements_avg_hh_size', 'Housing_Tenure_owner', 'Housing_Units_total', 'Housing_Units_vacancy_rate', 'Hous

In [2]:
unique_count = len(set(tract_list))
print("Unique tract count: " + str(unique_count))

Unique tract count: 204


In [3]:
tract_and_violations = prop_census_isd[['tract', 'isd_violations_count']]


#for each row in prop_census_isd
#

In [8]:
prop_census_isd_aggregate = prop_census_isd.groupby('tract').agg({'isd_violations_count': 'sum'})
print(prop_census_isd_aggregate)

# compute tract-level averages only on census demographic numeric fields (exclude property-specific columns)
property_cols = [c for c in prop_census_isd.columns if c.startswith('rs_prop_') or c.startswith('json_')]
key_cols = ['tract', 'isd_violations_count']

demographic_cols = [
    c for c in prop_census_isd.columns
    if c not in property_cols + key_cols
    and pd.api.types.is_numeric_dtype(prop_census_isd[c])
]

tract_averages = (
    prop_census_isd.groupby('tract')[demographic_cols]
    .mean()
    .rename(columns=lambda x: x + '_tract_avg')
)

# keep total violations as an aggregate field too
tract_aggregated = tract_averages.merge(
    prop_census_isd.groupby('tract', as_index=True)['isd_violations_count'].sum().rename('total_isd_violations'),
    left_index=True,
    right_index=True
)

# join tract-level statistics back onto the original property-level table
prop_census_isd_with_tract = prop_census_isd.merge(
    tract_aggregated.reset_index(),
    on='tract',
    how='left'
)

print('Result rows:', len(prop_census_isd_with_tract))
print(prop_census_isd_with_tract.filter(like='_tract_avg').head())

#write to csv
prop_census_isd_with_tract.to_csv("properties_census_isd_with_tract.csv", index=False)


        isd_violations_count
tract                       
101                       60
102                       33
201                       67
202                       84
301                       47
...                      ...
981300                     2
981501                     0
981700                     0
981800                     0
981900                     0

[204 rows x 1 columns]
Result rows: 367698
   Housing_Vehicles_share_no_vehicles_tract_avg  \
0                                      0.191520   
1                                      0.191520   
2                                      0.191520   
3                                      0.228362   
4                                      0.136259   

   Housing_LivingArrangements_household_pop_tract_avg  \
0                                          3657.0096    
1                                          3657.0096    
2                                          3657.0096    
3                                          5